# GPR Processing
**Lunar Leaper** -- Load stitched raw `.npz` data, tune processing parameters, and save the processed output.

Workflow: run cells top to bottom. Select a file, adjust sliders, then save.

In [5]:
import numpy as np
import json
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path
from scipy.ndimage import shift as ndshift

widgets.Widget.close_all()

## 1. Select data
Choose a stitched `_raw.npz` file. The sidecar `.json` is loaded automatically and provenance (stitch/patch info) is printed.

In [9]:
STITCH_DIR = Path('../../Data/GPR/Stitched')
PROC_DIR   = Path('../../Data/GPR/Processed')

def get_npz_files():
    return sorted(p.name for p in STITCH_DIR.glob('*_raw.npz'))

w_file = widgets.Dropdown(
    options=get_npz_files(),
    description='File:',
    style={'description_width': '50px'},
    layout=widgets.Layout(width='420px')
)

out_load = widgets.Output()

# Globals set by load_data(), read by processing and save cells
data = dist_axis = time_axis = info = None
sfreq = nyquist = freq = suggested_dewow = None

def load_data(_=None):
    global data, dist_axis, time_axis, info, sfreq, nyquist, freq, suggested_dewow
    out_load.clear_output(wait=True)
    with out_load:
        fname = w_file.value
        if not fname:
            print('No .npz files found in', STITCH_DIR)
            return
        npz_path  = STITCH_DIR / fname
        json_path = npz_path.with_suffix('.json')
        print(f'Loading {npz_path.name}...')

        npz       = np.load(str(npz_path))
        data      = npz['data'].astype(np.float64)
        dist_axis = npz['dist_axis']
        time_axis = npz['time_axis']

        with open(str(json_path), encoding='utf-8') as f:
            info = json.load(f)

        n_samples     = info['samples']
        time_window   = info['Total_time_window']
        sfreq         = n_samples / time_window * 1000
        nyquist       = sfreq / 2
        freq          = info['Freq']
        ns_per_sample = time_window / n_samples
        suggested_dewow = round(2 / freq * 1000 / ns_per_sample)

        print(f"Shape   : {data.shape[0]} samples x {data.shape[1]} traces")
        print(f"Distance: {dist_axis[0]:.1f} - {dist_axis[-1]:.1f} {info['Pos_units']}")
        print(f"Time    : {time_axis[0]:.1f} - {time_axis[-1]:.1f} ns")
        print(f"Freq    : {freq} MHz  |  sfreq: {sfreq:.0f} MHz  |  Nyquist: {nyquist:.0f} MHz")
        print(f"Suggested dewow: {suggested_dewow} samples")

        if info.get('stitch_info'):
            segs = ', '.join(s['line'] for s in info['stitch_info']['segments'])
            print(f"\nStitched from: {segs}")
        if info.get('patch_info'):
            srcs = [p['source'] for p in info['patch_info']['patches']]
            print(f"Patches applied: {srcs}")

        if 'update_sliders' in globals():
            update_sliders()

w_file.observe(load_data, names='value')
load_data()
display(w_file)
display(out_load)

Dropdown(description='File:', layout=Layout(width='420px'), options=('FlowerPetal1_50MHz_raw.npz', 'FlowerPeta…

Output()

## 2. Interactive processing
Adjust sliders to tune the radargram. When satisfied, save in section 3.

In [11]:
# -- Processing function -------------------------------------------------------
def apply_processing(dewow_w, tzero_shift, flow, fhigh, gain_exp):
    from gdp.preprocessing.filtering import dewow as dewow_fn, filter_data
    from gdp.preprocessing.gain import apply_gain as apply_gain_fn

    processed = data.copy()
    processed = dewow_fn(processed, window_length=dewow_w)

    if tzero_shift != 0:
        processed = ndshift(processed, (tzero_shift, 0), order=1, mode='constant', cval=0)

    try:
        processed = filter_data(processed, (flow, fhigh), sfreq, 'bandpass', N=4)
    except Exception as e:
        print(f'Bandpass failed: {e}')

    try:
        processed, _ = apply_gain_fn(processed, sfreq, 'linear', exponent=gain_exp)
    except Exception as e:
        print(f'Gain failed: {e}')

    return processed

# -- Widget initial values from loaded data ------------------------------------
_freq0 = freq if freq else 50
_nyq0  = nyquist if nyquist else 250
_dew0  = suggested_dewow if suggested_dewow else 15
_tz0   = -info['TZ_at_pt'] if info else 0.0
_d0    = [float(dist_axis[0]), float(dist_axis[-1])] if dist_axis is not None else [0.0, 100.0]

w_dewow = widgets.IntSlider(
    value=_dew0, min=1, max=150, step=1,
    description='Dewow (samples):',
    continuous_update=False,
    style={'description_width': '170px'}, layout=widgets.Layout(width='520px')
)
w_tzero = widgets.FloatSlider(
    value=_tz0, min=-200, max=100, step=0.01,
    description='Time zero shift (samples):',
    continuous_update=False,
    style={'description_width': '190px'}, layout=widgets.Layout(width='520px')
)
w_flow = widgets.FloatSlider(
    value=0.4 * _freq0, min=1, max=_freq0, step=1,
    description='Low cutoff (MHz):',
    continuous_update=False,
    style={'description_width': '170px'}, layout=widgets.Layout(width='520px')
)
w_fhigh = widgets.FloatSlider(
    value=min(2 * _freq0, _nyq0 * 0.95), min=_freq0, max=_nyq0 * 0.95, step=1,
    description='High cutoff (MHz):',
    continuous_update=False,
    style={'description_width': '170px'}, layout=widgets.Layout(width='520px')
)
w_gain = widgets.FloatSlider(
    value=2.5, min=0., max=3.5, step=0.1,
    description='Gain exponent:',
    continuous_update=False,
    style={'description_width': '170px'}, layout=widgets.Layout(width='520px')
)
w_clip = widgets.FloatSlider(
    value=90, min=50, max=100, step=0.5,
    description='Clip percentile (%):',
    continuous_update=False,
    style={'description_width': '170px'}, layout=widgets.Layout(width='520px'),
    readout_format='.1f'
)
w_xrange = widgets.FloatRangeSlider(
    value=_d0, min=_d0[0], max=_d0[1], step=0.1,
    description='X range (m):',
    continuous_update=False,
    style={'description_width': '170px'}, layout=widgets.Layout(width='520px'),
    readout_format='.1f'
)
w_depth_toggle = widgets.ToggleButton(
    value=False, description='Show depth', button_style='',
    layout=widgets.Layout(width='140px')
)
w_velocity = widgets.FloatSlider(
    value=0.13, min=0.05, max=0.3, step=0.005,
    description='Velocity (m/ns):',
    continuous_update=False,
    style={'description_width': '170px'}, layout=widgets.Layout(width='520px'),
    readout_format='.3f'
)

out = widgets.Output()
current_fig = None

def update_sliders():
    _f  = info['Freq']
    _nq = sfreq / 2
    w_flow.max    = _f
    w_flow.value  = 0.4 * _f
    w_fhigh.min   = 1
    w_fhigh.max   = _nq * 0.95
    w_fhigh.min   = _f
    w_fhigh.value = min(2 * _f, _nq * 0.95)
    w_tzero.value = -info['TZ_at_pt']
    w_dewow.value = suggested_dewow
    w_xrange.min  = dist_axis[0]
    w_xrange.max  = dist_axis[-1]
    w_xrange.value = [dist_axis[0], dist_axis[-1]]

# -- Plot ----------------------------------------------------------------------
def plot_radargram(dewow_w, tzero_shift, flow, fhigh, gain_exp,
                   show_depth, velocity, xrange, clip_pct):
    global current_fig
    if data is None:
        print('Load a file first.')
        return

    processed = apply_processing(dewow_w, tzero_shift, flow, fhigh, gain_exp)

    x0, x1   = xrange
    mask      = (dist_axis >= x0) & (dist_axis <= x1)
    dist_plot = dist_axis[mask]
    processed = processed[:, mask]

    vmax = np.percentile(np.abs(processed), clip_pct)

    if show_depth:
        y_axis  = time_axis * velocity / 2
        y_label = 'Depth (m)'
        y_hover = 'Depth: %{y:.2f} m'
    else:
        y_axis  = time_axis
        y_label = 'Two-way travel time (ns)'
        y_hover = 'Time: %{y:.1f} ns'

    fig = go.Figure(go.Heatmap(
        x=dist_plot, y=y_axis, z=processed,
        colorscale='Gray', zmin=-vmax, zmax=vmax,
        colorbar=dict(title='Ampl.', thickness=15, len=0.9),
        hovertemplate=(
            'Dist: %{x:.1f} m<br>' + y_hover +
            '<br>Amplitude: %{z:.2e}<extra></extra>'
        )
    ))

    stem = w_file.value.replace('_raw.npz', '')
    vel_str = f'  |  v = {velocity:.3f} m/ns' if show_depth else ''
    fig.update_layout(
        title=dict(
            text=(
                f'{stem}  |  Dewow: {dewow_w}  |  TZero: {tzero_shift:.2f}  |  '
                f'Filter: {flow:.0f}-{fhigh:.0f} MHz  |  Gain: {gain_exp}{vel_str}'
            ),
            font=dict(size=11)
        ),
        height=580,
        plot_bgcolor='white', paper_bgcolor='white',
        hovermode='closest',
        yaxis=dict(title=y_label, showgrid=False, autorange='reversed'),
        xaxis=dict(title=f"Distance ({info['Pos_units']})", showgrid=False)
    )

    current_fig = fig
    fig.show()

def on_change(_):
    out.clear_output(wait=True)
    with out:
        plot_radargram(
            int(w_dewow.value), float(w_tzero.value),
            float(w_flow.value), float(w_fhigh.value), float(w_gain.value),
            bool(w_depth_toggle.value), float(w_velocity.value),
            w_xrange.value, float(w_clip.value)
        )

for w in [w_dewow, w_tzero, w_flow, w_fhigh, w_gain,
          w_clip, w_xrange, w_depth_toggle, w_velocity]:
    w.observe(on_change, names='value')

display(widgets.VBox([
    widgets.HTML('<b>Processing</b>'),
    w_dewow, w_tzero, w_flow, w_fhigh, w_gain,
    widgets.HTML('<b>View</b>'),
    w_clip, w_xrange,
    widgets.HBox([w_depth_toggle, w_velocity]),
]))
display(out)
on_change(None)

Output()

## 3. Save
Save the processed array and current slider settings. Use **Load params** on a future session to restore exactly where you left off.

In [8]:
# -- Save processed data and parameters ----------------------------------------
# Processed .npz : data array + dist_axis + time_axis for downstream notebooks.
# Params .json   : all slider values, so processing can be reproduced or tweaked.
# Both saved to Data/GPR/Processed/ with the same stem as the source file.
# Load params: restores slider values from a previous save for the current file.

out_save = widgets.Output()

def _params():
    return {
        'source_file':     w_file.value,
        'dewow_window':    int(w_dewow.value),
        'tzero_shift':     float(w_tzero.value),
        'bandpass_low':    float(w_flow.value),
        'bandpass_high':   float(w_fhigh.value),
        'gain_exponent':   float(w_gain.value),
        'clip_percentile': float(w_clip.value),
        'velocity_mns':    float(w_velocity.value),
        'show_depth':      bool(w_depth_toggle.value),
    }

def _save_processed(_=None):
    out_save.clear_output(wait=True)
    with out_save:
        if data is None:
            print('No data loaded.')
            return
        stem = w_file.value.replace('_raw.npz', '')
        p    = _params()
        PROC_DIR.mkdir(parents=True, exist_ok=True)

        processed = apply_processing(
            p['dewow_window'], p['tzero_shift'],
            p['bandpass_low'], p['bandpass_high'],
            p['gain_exponent']
        )

        npz_path  = PROC_DIR / f'{stem}_processed.npz'
        json_path = PROC_DIR / f'{stem}_params.json'

        np.savez(str(npz_path), data=processed, dist_axis=dist_axis, time_axis=time_axis)
        with open(str(json_path), 'w', encoding='utf-8') as f:
            json.dump(p, f, indent=2)

        print(f'Saved .npz : {npz_path.resolve()}')
        print(f'Saved .json: {json_path.resolve()}')
        print(f'Array shape: {processed.shape}')

def _load_params(_=None):
    out_save.clear_output(wait=True)
    with out_save:
        stem      = w_file.value.replace('_raw.npz', '')
        json_path = PROC_DIR / f'{stem}_params.json'
        if not json_path.exists():
            print(f'No saved params found: {json_path}')
            return
        with open(str(json_path), encoding='utf-8') as f:
            p = json.load(f)

        w_dewow.value  = p['dewow_window']
        w_tzero.value  = p['tzero_shift']
        w_fhigh.min    = 1
        w_fhigh.max    = nyquist * 0.95
        w_fhigh.min    = p['bandpass_low']
        w_fhigh.value  = p['bandpass_high']
        w_flow.max     = p['bandpass_low']
        w_flow.value   = p['bandpass_low']
        w_gain.value   = p['gain_exponent']
        w_clip.value   = p.get('clip_percentile', 90)
        w_velocity.value = p.get('velocity_mns', 0.13)
        print(f"Loaded params from {json_path.name}")

btn_save    = widgets.Button(description='Save processed', button_style='success', layout=widgets.Layout(width='150px'))
btn_load_p  = widgets.Button(description='Load params',    button_style='info',    layout=widgets.Layout(width='130px'))
btn_save.on_click(_save_processed)
btn_load_p.on_click(_load_params)

display(widgets.VBox([
    widgets.HTML('<b>Save / load params</b>'),
    widgets.HBox([btn_save, btn_load_p]),
    out_save,
]))